In [1]:
import pandas as pd
df = pd.read_csv('/home/bhavish-berry/PycharmProjects/ai_training_colllege_first_model/data/movielens_engineered.csv')
df.head()

,movieId,title,genres,avg_rating,rating_count,all_tags,imdbId,tmdbId,year,num_genres,...,genre_Horror,genre_IMAX,genre_Musical,genre_Mystery,genre_Romance,genre_Sci-Fi,genre_Thriller,genre_War,genre_Western,tag_count
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,3.920930,215.0,fun pixar,114709,862.0,1995.0,5,...,0,0,0,0,0,0,0,0,0,2
1,2,Jumanji (1995),Adventure|Children|Fantasy,3.431818,110.0,Robin Williams fantasy game magic board game,113497,8844.0,1995.0,3,...,0,0,0,0,0,0,0,0,0,7
2,3,Grumpier Old Men (1995),Comedy|Romance,3.259615,52.0,moldy old,113228,15602.0,1995.0,2,...,0,0,0,0,1,0,0,0,0,2
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance,2.357143,7.0,NaN,114885,31357.0,1995.0,3,...,0,0,0,0,1,0,0,0,0,0
4,5,Father of the Bride Part II (1995),Comedy,3.071429,49.0,pregnancy remake,113041,11862.0,1995.0,1,...,0,0,0,0,0,0,0,0,0,2


## Data Preprocessing

`movieId` is an arbitrary identifier with no learnable relationship to the
features, and `title`/`genres` (raw)/`imdbId`/`tmdbId` are identifiers or
text we won't feed into a model. We'll instead build a meaningful
classification target: whether a movie is **highly rated**
(`avg_rating >= 3.5`), using the one-hot genres, year, and tag/rating
volume as features.

In [2]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 9742 entries, 0 to 9741
Data columns (total 30 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   movieId            9742 non-null   int64  
 1   title              9742 non-null   str    
 2   genres             9742 non-null   str    
 3   avg_rating         9724 non-null   float64
 4   rating_count       9724 non-null   float64
 5   all_tags           1572 non-null   str    
 6   imdbId             9742 non-null   int64  
 7   tmdbId             9734 non-null   float64
 8   year               9729 non-null   float64
 9   num_genres         9742 non-null   int64  
 10  genre_Action       9742 non-null   int64  
 11  genre_Adventure    9742 non-null   int64  
 12  genre_Animation    9742 non-null   int64  
 13  genre_Children     9742 non-null   int64  
 14  genre_Comedy       9742 non-null   int64  
 15  genre_Crime        9742 non-null   int64  
 16  genre_Documentary  9742 non-null   

In [4]:
df.isnull().sum()

movieId                 0
title                   0
genres                  0
avg_rating             18
rating_count           18
all_tags             8170
imdbId                  0
tmdbId                  8
year                   13
num_genres              0
genre_Action            0
genre_Adventure         0
genre_Animation         0
genre_Children          0
genre_Comedy            0
genre_Crime             0
genre_Documentary       0
genre_Drama             0
genre_Fantasy           0
genre_Film-Noir         0
genre_Horror            0
genre_IMAX              0
genre_Musical           0
genre_Mystery           0
genre_Romance           0
genre_Sci-Fi            0
genre_Thriller          0
genre_War               0
genre_Western           0
tag_count               0
dtype: int64

In [7]:
df_clean = df.dropna(subset=["avg_rating", "rating_count"]).copy()
df_clean["year"] = df_clean["year"].fillna(df_clean["year"].median())
df_clean["tag_count"] = df_clean["tag_count"].fillna(0)

print(df_clean.shape)
df_clean.isna().sum()

(9724, 30)


movieId                 0
title                   0
genres                  0
avg_rating              0
rating_count            0
all_tags             8170
imdbId                  0
tmdbId                  8
year                    0
num_genres              0
genre_Action            0
genre_Adventure         0
genre_Animation         0
genre_Children          0
genre_Comedy            0
genre_Crime             0
genre_Documentary       0
genre_Drama             0
genre_Fantasy           0
genre_Film-Noir         0
genre_Horror            0
genre_IMAX              0
genre_Musical           0
genre_Mystery           0
genre_Romance           0
genre_Sci-Fi            0
genre_Thriller          0
genre_War               0
genre_Western           0
tag_count               0
dtype: int64

In [8]:
# Binary classification target: is this movie highly rated?
df_clean["high_rating"] = (df_clean["avg_rating"] >= 3.5).astype(int)
df_clean["high_rating"].value_counts(normalize=True)

high_rating
0    0.518717
1    0.481283
Name: proportion, dtype: float64

In [9]:
genre_cols = [c for c in df_clean.columns if c.startswith("genre_")]
feature_cols = genre_cols + ["year", "num_genres", "tag_count", "rating_count"]

X = df_clean[feature_cols]
y = df_clean["high_rating"]

X.head()

,genre_Action,genre_Adventure,genre_Animation,genre_Children,genre_Comedy,genre_Crime,genre_Documentary,genre_Drama,genre_Fantasy,genre_Film-Noir,...,genre_Mystery,genre_Romance,genre_Sci-Fi,genre_Thriller,genre_War,genre_Western,year,num_genres,tag_count,rating_count
0,0,1,1,1,1,0,0,0,1,0,...,0,0,0,0,0,0,1995.0,5,2,215.0
1,0,1,0,1,0,0,0,0,1,0,...,0,0,0,0,0,0,1995.0,3,7,110.0
2,0,0,0,0,1,0,0,0,0,0,...,0,1,0,0,0,0,1995.0,2,2,52.0
3,0,0,0,0,1,0,0,1,0,0,...,0,1,0,0,0,0,1995.0,3,0,7.0
4,0,0,0,0,1,0,0,0,0,0,...,0,0,0,0,0,0,1995.0,1,2,49.0


In [10]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# SVM is sensitive to feature scale, so we standardize.
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

X_train_scaled.shape, X_test_scaled.shape

((7779, 23), (1945, 23))

## Train an SVM Classifier

In [11]:
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

svm_model = SVC(kernel="rbf", C=1.0, gamma="scale", random_state=42)
svm_model.fit(X_train_scaled, y_train)

y_pred = svm_model.predict(X_test_scaled)

Accuracy: 0.6663239074550128

Classification Report:
               precision    recall  f1-score   support

           0       0.67      0.69      0.68      1009
           1       0.66      0.64      0.65       936

    accuracy                           0.67      1945
   macro avg       0.67      0.67      0.67      1945
weighted avg       0.67      0.67      0.67      1945


Confusion Matrix:
 [[701 308]
 [341 595]]


## Compare Kernels

SVM performance depends heavily on the kernel choice. Let's compare
linear, rbf, and poly kernels on the same scaled data.

In [12]:
for kernel in ["linear", "rbf", "poly"]:
    model = SVC(kernel=kernel, random_state=42)
    model.fit(X_train_scaled, y_train)
    acc = accuracy_score(y_test, model.predict(X_test_scaled))
    print(f"{kernel:>8} kernel accuracy: {acc:.4f}")

  linear kernel accuracy: 0.6257
     rbf kernel accuracy: 0.6663
    poly kernel accuracy: 0.6560
